# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
# Cargamos los datos preparados en el notebook anterior al instante
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval n_features

6-element Vector{Symbol}:
 :df_trainval
 :df_test
 :X_trainval
 :y_trainval
 :folds_trainval
 :n_features

# Modelos básicos y selección de atributos (20%)

In [4]:
# Definición de diccionarios de configuración con MLP corregido
dic_filtros = Dict(
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => nothing,
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

# Modelos con capas ocultas correctamente definidas
dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => SVC(cost=0.1),
    "SVM_0.5" => SVC(cost=0.5),
    "SVM_1" => SVC(cost=1.0)
);

In [ ]:
results_df = DataFrame(
    Filter = String[], Reduction = String[], Model = String[],
    Accuracy = Float64[], F1_Score = Float64[]
)
measures = [accuracy, multiclass_f1score]
output_file = "resultados_parciales.csv"

for (filt_name, filt_model) in dic_filtros
    for (red_name, red_model) in dic_reducciones
        for (mod_name, mod_model) in dic_modelos
            
            println(">>> Evaluando: $filt_name + $red_name + $mod_name")
            
            # Usamos el ManualPipeline del backend
            scaler = MyMinMaxScaler()
            pipe = ManualPipeline(scaler, filt_model, red_model, mod_model)
            
            try
                evaluation = evaluate!(
                    pipe, X_trainval, y_trainval,
                    resampling = folds_trainval, measures = measures, verbosity = 0
                )
                
                acc, f1 = evaluation.measurement[1], evaluation.measurement[2]
                
                push!(results_df, (filt_name, red_name, mod_name, acc, f1))
                println("    Resultado: Acc=$acc | F1=$f1")
                
                CSV.write(output_file, results_df) # Guardado continuo
                
            catch e
                println("!!! Error: $e")
                push!(results_df, (filt_name, red_name, mod_name, NaN, NaN))
            end
        end
    end
end

pretty_table(results_df)

Iniciando ejecución...
>>> Evaluando: MI + ICA + NeuralNetwork_100


Excessive output truncated after 10485828 bytes.